# STATISTIQUES

## IMPORTS

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as spark_func

## Création de la session Spark et récupération du dataset

In [ ]:
path = "data/gaming_mental_health_merged.csv"
#path = "hdfs://localhost:9000/projet/gaming_mental_health/input/gaming_mental_health.csv"

spark = (
    SparkSession.builder
    .appName("MonPremierProjet")
    .master("local[*]")
    .getOrCreate()
)

df = spark.read.csv(
    path,
    header=True,
    inferSchema=True
)

## Création des différentes statistiques

### Joueur le plus dépensier du dataset

In [ ]:
money_spender = (df.select(df.gender, df.primary_game, df.monthly_game_spending_usd)
)

money_spender.sort(df.monthly_game_spending_usd, ascending = False).show(1)

### Âge moyen des personnes du dataset

In [ ]:
mean_age = (df.select(df.age)
            .agg(spark_func.round(spark_func.mean(df.age),1).alias("Average age"))
)

print("Average age in the dataset :")
mean_age.show()

### Proportion sexes par genre de jeu

In [ ]:
gender_count_by_game_genre = df.groupBy(df.game_genre, df.gender).count()

people_by_game_genre = gender_count_by_game_genre.groupBy(df.game_genre).agg(spark_func.sum("count").alias("total"))

result1 = (
    gender_count_by_game_genre.join(people_by_game_genre, on="game_genre")
          .withColumn("proportion(%)", spark_func.round((spark_func.col("count") / spark_func.col("total"))*100, 1))
)

print("Female/Male/Other proportions depending on game genres :")
result1.orderBy("game_genre", "gender").show(result1.count())

In [ ]:
print("Genres where women are most present (top 5) :")
result2 = result1.select("game_genre", "gender", "proportion(%)").filter(result1.gender=="Female")
result2.sort("proportion(%)",ascending = False).show(5)

### Dépenses moyennes par sexe

In [ ]:
mean_spending_per_gender = (
    df.select(df.gender, df.monthly_game_spending_usd)
    .groupBy(df.gender)
    .agg(spark_func.round(spark_func.avg(df.monthly_game_spending_usd),2).alias("Mean spending per month"))
)

print("Average spending on video games per month per gender :")
mean_spending_per_gender.show()

### Moyenne d'heure de sommeil par jeu

In [ ]:
mean_sleep_hours_per_game = (
    df.select(df.primary_game, df.sleep_hours)
    .groupBy(df.primary_game)
    .agg(spark_func.round(spark_func.avg(df.sleep_hours),2).alias("Average sleep hours"))
)

print("Average hours of sleep per main game played :")
mean_sleep_hours_per_game.sort("Average sleep hours").show(mean_sleep_hours_per_game.count())

### Plus vieilles bases de joueurs

In [ ]:
oldest_playerbase = (df.select(df.primary_game, df.age)
                    .groupBy(df.primary_game)
                    .agg(spark_func.round(spark_func.avg(df.age),2).alias("Average age"))
)

print("Top 5 oldest playerbases :")
oldest_playerbase.sort("Average age", ascending = False).show(5)

### Moyenne des heures de sommeil par jeu

In [ ]:
sleep_per_game = (
    df.select(df.primary_game,df.sleep_hours)
    .groupBy(df.primary_game)
    .agg(spark_func.round(spark_func.avg(df.sleep_hours),2).alias("Average sleep hours"))
)

print("Average sleep hours per main game :")
sleep_per_game.sort("Average sleep hours", ascending = True).show(sleep_per_game.count())